# 03 · Markov Decision Processes
**FOUNDATIONS** · ⏱ 45 min · 🧪 3 exercises · ⭐ 350 XP

*The formal model behind RL — states, dynamics, policies, value functions and the Bellman equations.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "03-mdps"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

In the bandit lesson, actions had no consequences beyond their immediate reward. Real problems
are not like that: a chess move changes the board, a robot's step changes where it stands, and
a good move now may only pay off twenty moves later. The **Markov decision process (MDP)** is
the mathematical model of that situation, and almost every RL algorithm is defined in its
terms.

This lesson is the most mathematical in the course. Take it slowly: everything you build in
the next several lessons (dynamic programming, Monte Carlo, TD, Q-learning, DQN) is an
approximate way of solving the two equations at the end of it.

## 01 · The Markov property

A state is **Markov** if it captures everything about the past that matters for the future:

```text
Pr{ Sₜ₊₁, Rₜ₊₁ | Sₜ, Aₜ }  =  Pr{ Sₜ₊₁, Rₜ₊₁ | S₀, A₀, R₁, …, Sₜ, Aₜ }
```

Once you know the current state and action, the history adds nothing. A chess position is
Markov (castling rights and the move counter included). A single frame of a video game usually
is **not**: one frame tells you where the ball is, not which way it is moving, which is why
DQN stacks the last four frames.

> **ℹ️ NOTE** · **Markov is a property of your state representation**
>
> The same world can be Markov or not depending on what you put in the state. Positions alone
> are not Markov for a bouncing ball; positions and velocities are. Much of practical RL is
> choosing a state that is (close enough to) Markov.

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("03-mdps#0", {"q": "A robot's state is just its (x, y) position, but it moves with momentum. Is that state Markov?", "options": ["Yes, position is all that matters", "No, the next position also depends on velocity, which the state omits", "Only if rewards are deterministic", "Markov only applies to discrete states"], "answer": 1, "explain": "Two visits to the same (x, y) with different velocities lead to different futures, so the past (which reveals velocity) still matters. Add velocity to the state to restore the Markov property."})

## 02 · The MDP

A finite MDP is the tuple **(S, A, p, γ)**:

- **S**, a finite set of states, and **A**, a finite set of actions
- **p**, the **dynamics**: the probability of each next state and reward
- **γ ∈ [0, 1]**, the discount factor from the previous lesson

```text
p(s', r | s, a) = Pr{ Sₜ₊₁ = s', Rₜ₊₁ = r | Sₜ = s, Aₜ = a }
```

Because it is a probability distribution, Σₛ' Σᵣ p(s', r | s, a) = 1 for every (s, a). From the
four-argument p you can derive everything else you might want:

```text
p(s' | s, a) = Σᵣ p(s', r | s, a)          r(s, a) = Σₛ' Σᵣ r · p(s', r | s, a)
```

In this course's `GridWorld`, p is stored literally. `env.P[s][a]` is the list of outcomes
`(prob, next_state, reward, terminated)`, one entry per possible (s', r):

In [ ]:
# 📎 Example from the briefing — run it, change it, break it.
from rlenvs import GridWorld
env = GridWorld(["...G",
                 ".#.X",
                 "S..."], slip=0.2, step_reward=-0.04)
env.P[env.state_of(1, 2)][0]     # try to go UP from the cell below the goal row
# [(0.8, 2, -0.04, False),  (0.1, 7, -1.0, True),  (0.1, 6, -0.04, False)]
#  went up                  slipped right: the pit   slipped left: bumped the wall, stayed

> **🔑 KEY IDEA** · **Everything about the world is in p**
>
> An MDP is fully specified by its dynamics. **Planning** methods (next lesson) are handed p and
> compute good behaviour from it. **Learning** methods (the rest of the course) never see p:
> they only sample from it by acting.

Terminal states such as G and X end the episode. By convention their value is 0: no reward ever
follows them. The `terminated` flag tells you when a transition enters one.

## 03 · Policies and value functions

A **policy** π(a | s) is the probability of taking action *a* in state *s*. A deterministic
policy puts all its probability on one action per state. The two value functions from lesson 1,
now in full:

```text
v_π(s) = E_π[ Gₜ | Sₜ = s ]            q_π(s, a) = E_π[ Gₜ | Sₜ = s, Aₜ = a ]
```

v_π answers "how good is it to be here, if I then follow π?". q_π answers "how good is it to
take *a* here and follow π afterwards?". They are tied together by two one-line identities you
should know by heart:

```text
v_π(s) = Σₐ π(a|s) · q_π(s, a)
q_π(s, a) = Σₛ',ᵣ p(s', r | s, a) · [ r + γ · v_π(s') ]
```

The first averages over **your** choice (the policy). The second averages over the
**world's** response (the dynamics). The second one is called a **one-step lookahead**: it
turns state values into action values, and acting greedily on those action values turns values
into a policy.

Worked example: corridor `S . G`, step reward −1, entering G pays +10, γ = 0.5, and suppose
V = [2, 5, 0]. From S, RIGHT reaches the middle: q(S, RIGHT) = −1 + 0.5·5 = 1.5. The other three
actions bump the wall and stay: q = −1 + 0.5·2 = 0. From the middle, RIGHT enters the terminal
goal: q = 10, with no γ·V term because nothing follows termination.

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("03-mdps#1", {"q": "In the corridor above, what is q(middle, LEFT)?", "options": ["−1 + 0.5·2 = 0", "−1 + 0.5·5 = 1.5", "10", "−1"], "answer": 0, "explain": "LEFT from the middle leads to S with reward −1, so q = −1 + γ·V(S) = −1 + 0.5·2 = 0."})

---
### 🧪 Exercise 1 · One-step lookahead · ⭐ **100 XP**

Turn state values into action values, and action values into a policy.

1. `q_from_v(env, V, s, gamma)` — return an array with, for each action *a*,
   `Σ prob · (r + γ · V[s'])` over the outcomes `(prob, s', r, terminated)` in `env.P[s][a]`. When `terminated` is True the episode is over: use just `r`.
2. `q_table(env, V, gamma)` — stack `q_from_v` for every state into an `(n_states, n_actions)` array.
3. `greedy_policy(Q)` — the argmax action of each row.

This lookahead is the inner loop of every planning algorithm in the next lesson. **Run** shows a value map with its greedy policy, and the four q-values at the start state.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 03-mdps/01-one-step-lookahead
import numpy as np
import rlhud
from rlenvs import GridWorld, ACTION_NAMES


def q_from_v(env, V, s, gamma):
    """One-step lookahead: the value of each action in state s, given state values V.

        q(s, a) = Σ_{s', r} p(s', r | s, a) · [ r + γ · V(s') ]

    env.P[s][a] is a list of (prob, next_state, reward, terminated).
    A transition that terminates has no future: do not add γ·V(s') for it.
    Returns a NumPy array of length env.n_actions.
    """
    q = np.zeros(env.n_actions)
    # TODO: loop over actions and their outcomes in env.P[s][a]
    raise NotImplementedError
    return q


def q_table(env, V, gamma):
    """q_from_v for every state: an (n_states, n_actions) array."""
    # TODO
    raise NotImplementedError


def greedy_policy(Q):
    """For each state, the action with the largest q-value (ties → lowest index)."""
    # TODO: one line with np.argmax
    raise NotImplementedError


if __name__ == "__main__":
    env = GridWorld(["...G",
                     ".#.X",
                     "S..."], step_reward=-0.04)
    gamma = 0.9
    # The optimal values v*(s) of this world (you will compute them yourself with DP next
    # lesson). Values alone don't say what to DO — the one-step lookahead does.
    V = np.array([0.734, 0.860, 1.000, 0.0,
                  0.621, 0.000, 0.860, 0.0,
                  0.519, 0.621, 0.734, 0.621])
    Q = q_table(env, V, gamma)
    pi = greedy_policy(Q)
    rlhud.grid(env.to_grid(V), title="Given V(s) with the greedy policy",
               policy=env.policy_grid(pi), walls=env.walls_mask(), labels=env.labels())
    s = env.start
    rlhud.bars(Q[s], labels=ACTION_NAMES, title=f"q(start, a) — one-step lookahead from state {s}",
               highlight=int(pi[s]))
    rlhud.table([[ACTION_NAMES[a], round(float(Q[s, a]), 4)] for a in range(4)],
                headers=["action", "q(start, a)"], title="Lookahead at the start state")

In [ ]:
#@title ✅ **Check** · One-step lookahead (+100 XP) — tests the code you last ran above
rlcolab.check("03-mdps/01-one-step-lookahead")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

`env.P[s][a]` is a short list of tuples. Two nested loops are enough: over `a in range(env.n_actions)`, then `for prob, s2, r, done in env.P[s][a]`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Accumulate `q[a] += prob * (r + gamma * V[s2])`, but replace the `gamma * V[s2]` part with 0 when `done` is True. Terminal states themselves only have self-loops with reward 0 and done=True, so their q is 0 automatically.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

`q[a] += prob * (r + (0.0 if done else gamma * V[s2]))`; `q_table` is `np.array([q_from_v(env, V, s, gamma) for s in range(env.n_states)])`; `greedy_policy` is `np.argmax(Q, axis=1)`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("03-mdps/01-one-step-lookahead")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 03-mdps/01-one-step-lookahead
# Reference solution
import numpy as np
import rlhud
from rlenvs import GridWorld, ACTION_NAMES


def q_from_v(env, V, s, gamma):
    """One-step lookahead: the value of each action in state s, given state values V.

        q(s, a) = Σ_{s', r} p(s', r | s, a) · [ r + γ · V(s') ]

    env.P[s][a] is a list of (prob, next_state, reward, terminated).
    A transition that terminates has no future: do not add γ·V(s') for it.
    Returns a NumPy array of length env.n_actions.
    """
    q = np.zeros(env.n_actions)
    for a in range(env.n_actions):
        for prob, s2, r, done in env.P[s][a]:
            q[a] += prob * (r + (0.0 if done else gamma * V[s2]))
    return q


def q_table(env, V, gamma):
    """q_from_v for every state: an (n_states, n_actions) array."""
    return np.array([q_from_v(env, V, s, gamma) for s in range(env.n_states)])


def greedy_policy(Q):
    """For each state, the action with the largest q-value (ties → lowest index)."""
    return np.argmax(np.asarray(Q), axis=1)


if __name__ == "__main__":
    env = GridWorld(["...G",
                     ".#.X",
                     "S..."], step_reward=-0.04)
    gamma = 0.9
    # The optimal values v*(s) of this world (you will compute them yourself with DP next
    # lesson). Values alone don't say what to DO — the one-step lookahead does.
    V = np.array([0.734, 0.860, 1.000, 0.0,
                  0.621, 0.000, 0.860, 0.0,
                  0.519, 0.621, 0.734, 0.621])
    Q = q_table(env, V, gamma)
    pi = greedy_policy(Q)
    rlhud.grid(env.to_grid(V), title="Given V(s) with the greedy policy",
               policy=env.policy_grid(pi), walls=env.walls_mask(), labels=env.labels())
    s = env.start
    rlhud.bars(Q[s], labels=ACTION_NAMES, title=f"q(start, a) — one-step lookahead from state {s}",
               highlight=int(pi[s]))
    rlhud.table([[ACTION_NAMES[a], round(float(Q[s, a]), 4)] for a in range(4)],
                headers=["action", "q(start, a)"], title="Lookahead at the start state")

## 04 · The Bellman expectation equation

Substitute one identity into the other and you get a recursive equation for v_π alone:

```text
v_π(s) = Σₐ π(a|s) Σₛ',ᵣ p(s', r | s, a) [ r + γ v_π(s') ]
```

It says: *the value of a state is the expected immediate reward plus the discounted value of
wherever you land*, which is just Gₜ = Rₜ₊₁ + γ Gₜ₊₁ with expectations taken.

> **ℹ️ NOTE** · **Backup diagrams, in words**
>
> Picture the v_π equation as a two-level tree. At the root is an open circle, state *s*. From it
> hang one branch per action, each ending in a solid dot (a state–action pair), weighted by
> π(a|s). From each dot hang one branch per outcome (s', r), weighted by p, each ending in an open
> circle s'. Computing v_π(s) "backs up" the values from the leaves to the root: average over the
> outcome branches, then average over the action branches. The q_π diagram is the same tree
> started from a solid dot: outcomes first, then the next action.

### Solving it as a linear system

For a fixed π, there is one Bellman equation per state, and each one is **linear** in the
unknown values. Collect them into vectors. Let r_π(s) be the expected immediate reward under
π and P_π(s, s') the probability of moving from s to a non-terminal s':

```text
v_π = r_π + γ P_π v_π      ⇔      (I − γ P_π) v_π = r_π
```

That is an ordinary `n × n` linear system, `np.linalg.solve(I − γ P_π, r_π)`. With γ < 1 the
matrix is always invertible; with γ = 1 it is invertible as long as π reaches a terminal state
with probability 1 from everywhere.

Worked example: corridor `S . G`, reward only for entering G (+1), uniform random policy, γ = 0.9.
From S, one action in four moves right; the rest stay. From the middle, RIGHT ends the
episode with +1, LEFT goes back, UP/DOWN stay:

```text
v(S) = 0.9 · ( ¾ v(S) + ¼ v(M) )
v(M) = ¼ · 1 + 0.9 · ( ¼ v(S) + ½ v(M) )
```

Solving gives v(S) ≈ 0.439 and v(M) ≈ 0.634. Your code will produce exactly these numbers.

In [ ]:
#@title 🧠 **Knowledge check 3** · +25 XP — run, then pick an answer
rlcolab.quiz("03-mdps#2", {"q": "Why does solving v_π with a linear system NOT scale to large problems?", "options": ["Bellman equations are nonlinear", "It needs the full model p, and solving costs about n³ for n states", "It only works for deterministic policies", "It only works for γ = 1"], "answer": 1, "explain": "Direct solution costs O(n³) and needs P_π explicitly. Backgammon has ~10²⁰ states. Iterative methods (next lesson) and then sampling methods (later) get around both problems."})

---
### 🧪 Exercise 2 · Solving the Bellman equation exactly · ⭐ **130 XP**

For a **fixed** policy, the Bellman expectation equation is a system of linear equations, one per state. Solve it.

1. `policy_model(env, policy)` — with `policy[s, a] = π(a|s)`, build
   - `P_pi[s, s'] = Σₐ π(a|s) · p(s'|s,a)` counting **only transitions that do not terminate**
   - `r_pi[s] = Σₐ π(a|s) · Σ p · r` (expected immediate reward, terminating transitions included)
2. `solve_v(env, policy, gamma)` — solve `(I − γ P_π) v = r_π` with `np.linalg.solve`.

Dropping terminating transitions from `P_π` is what makes terminal states worth 0 and keeps the system solvable even with γ = 1. **Run** shows v_π for the random policy and for a hand-written one, and how v_π(start) depends on γ.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 03-mdps/02-solve-bellman
import numpy as np
import rlhud
from rlenvs import GridWorld


def policy_model(env, policy):
    """Collapse the MDP under a fixed policy into a Markov reward process.

    policy[s, a] = π(a | s)   (an n_states × n_actions array of probabilities)

    Returns (P_pi, r_pi):
        P_pi[s, s'] = Σ_a π(a|s) · p(s' | s, a)      — only NON-terminating transitions
        r_pi[s]     = Σ_a π(a|s) · Σ p · r            — expected immediate reward
    Transitions with terminated=True contribute their reward but no successor,
    so terminal (and wall) rows of P_pi stay all-zero.
    """
    n = env.n_states
    P_pi = np.zeros((n, n))
    r_pi = np.zeros(n)
    # TODO: loop over states, actions and outcomes; accumulate π(a|s)·prob into
    #       r_pi (times r) and P_pi (only when the transition does not terminate)
    raise NotImplementedError
    return P_pi, r_pi


def solve_v(env, policy, gamma):
    """Exact v_π from the Bellman expectation equation in matrix form:

        v = r_π + γ P_π v    ⇔    (I − γ P_π) v = r_π
    """
    # TODO: build the model, then one call to np.linalg.solve
    raise NotImplementedError


def uniform_policy(env):
    return np.full((env.n_states, env.n_actions), 1.0 / env.n_actions)


if __name__ == "__main__":
    env = GridWorld(["...G",
                     ".#.X",
                     "S..."], slip=0.2, step_reward=-0.04)
    gamma = 0.9
    v_rand = solve_v(env, uniform_policy(env), gamma)
    rlhud.grid(env.to_grid(v_rand), title="v_π for the uniform random policy (γ = 0.9, slip 0.2)",
               walls=env.walls_mask(), labels=env.labels())

    # A hand-written policy: go up the left side, then right along the top.
    UP, RIGHT, LEFT = 0, 1, 3
    acts = np.full(env.n_states, RIGHT)
    for r, c in [(2, 0), (1, 0), (1, 2), (2, 2)]:
        acts[env.state_of(r, c)] = UP
    acts[env.state_of(2, 3)] = LEFT
    det = np.eye(4)[acts]
    v_det = solve_v(env, det, gamma)
    rlhud.grid(env.to_grid(v_det), title="v_π for a hand-written policy",
               policy=env.policy_grid(acts), walls=env.walls_mask(), labels=env.labels())
    rlhud.stat("v_random(start)", float(v_rand[env.start]))
    rlhud.stat("v_hand(start)", float(v_det[env.start]))

    # How does far-sightedness change the value of the start state?
    gammas = np.linspace(0.0, 0.99, 34)
    rlhud.plot([float(solve_v(env, det, g)[env.start]) for g in gammas], x=gammas.tolist(),
               label="hand-written policy", title="v_π(start) as γ varies", xlabel="γ")

In [ ]:
#@title ✅ **Check** · Solving the Bellman equation exactly (+130 XP) — tests the code you last ran above
rlcolab.check("03-mdps/02-solve-bellman")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Three nested loops: states `s`, actions `a`, and outcomes `(prob, s2, r, done)` in `env.P[s][a]`. The weight of an outcome under the policy is `policy[s, a] * prob`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Every outcome adds `w * r` to `r_pi[s]`. Only outcomes with `done == False` add `w` to `P_pi[s, s2]`. Terminal and wall states only have `done=True` self-loops, so their rows stay zero and their value comes out 0.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

`solve_v`: `P_pi, r_pi = policy_model(env, policy); return np.linalg.solve(np.eye(env.n_states) - gamma * P_pi, r_pi)`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("03-mdps/02-solve-bellman")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 03-mdps/02-solve-bellman
# Reference solution
import numpy as np
import rlhud
from rlenvs import GridWorld


def policy_model(env, policy):
    """Collapse the MDP under a fixed policy into a Markov reward process.

    policy[s, a] = π(a | s)   (an n_states × n_actions array of probabilities)

    Returns (P_pi, r_pi):
        P_pi[s, s'] = Σ_a π(a|s) · p(s' | s, a)      — only NON-terminating transitions
        r_pi[s]     = Σ_a π(a|s) · Σ p · r            — expected immediate reward
    Transitions with terminated=True contribute their reward but no successor,
    so terminal (and wall) rows of P_pi stay all-zero.
    """
    n = env.n_states
    P_pi = np.zeros((n, n))
    r_pi = np.zeros(n)
    for s in range(n):
        for a in range(env.n_actions):
            pa = policy[s, a]
            if pa == 0:
                continue
            for prob, s2, r, done in env.P[s][a]:
                r_pi[s] += pa * prob * r
                if not done:
                    P_pi[s, s2] += pa * prob
    return P_pi, r_pi


def solve_v(env, policy, gamma):
    """Exact v_π from the Bellman expectation equation in matrix form:

        v = r_π + γ P_π v    ⇔    (I − γ P_π) v = r_π
    """
    P_pi, r_pi = policy_model(env, policy)
    return np.linalg.solve(np.eye(env.n_states) - gamma * P_pi, r_pi)


def uniform_policy(env):
    return np.full((env.n_states, env.n_actions), 1.0 / env.n_actions)


if __name__ == "__main__":
    env = GridWorld(["...G",
                     ".#.X",
                     "S..."], slip=0.2, step_reward=-0.04)
    gamma = 0.9
    v_rand = solve_v(env, uniform_policy(env), gamma)
    rlhud.grid(env.to_grid(v_rand), title="v_π for the uniform random policy (γ = 0.9, slip 0.2)",
               walls=env.walls_mask(), labels=env.labels())

    # A hand-written policy: go up the left side, then right along the top.
    UP, RIGHT, LEFT = 0, 1, 3
    acts = np.full(env.n_states, RIGHT)
    for r, c in [(2, 0), (1, 0), (1, 2), (2, 2)]:
        acts[env.state_of(r, c)] = UP
    acts[env.state_of(2, 3)] = LEFT
    det = np.eye(4)[acts]
    v_det = solve_v(env, det, gamma)
    rlhud.grid(env.to_grid(v_det), title="v_π for a hand-written policy",
               policy=env.policy_grid(acts), walls=env.walls_mask(), labels=env.labels())
    rlhud.stat("v_random(start)", float(v_rand[env.start]))
    rlhud.stat("v_hand(start)", float(v_det[env.start]))

    # How does far-sightedness change the value of the start state?
    gammas = np.linspace(0.0, 0.99, 34)
    rlhud.plot([float(solve_v(env, det, g)[env.start]) for g in gammas], x=gammas.tolist(),
               label="hand-written policy", title="v_π(start) as γ varies", xlabel="γ")

You can watch values form under a fixed policy in the widget below. It applies the Bellman
expectation equation repeatedly to the uniform random policy, the iterative alternative to the
linear solve that the next lesson develops.

In [ ]:
#@title 🎮 **Interactive** · Dynamic programming sandbox
rlcolab.widget("gridworld", map=["...G", ".#.X", "S..."], gamma=0.9, step=-0.04, slip=0.2, mode="policy-evaluation")

## 05 · Optimal value functions

Value functions impose an ordering on policies: π ≥ π' if v_π(s) ≥ v_π'(s) **in every
state**. A remarkable fact about finite MDPs is that there is always at least one policy that is
better than or equal to every other in every state at once. That is an **optimal policy** π*,
and all optimal policies share the same value functions:

```text
v*(s) = maxπ v_π(s)            q*(s, a) = maxπ q_π(s, a)
```

### The Bellman optimality equations

An optimal policy picks the best action, not an average one, so the "average over the policy"
step becomes a max:

```text
v*(s) = maxₐ Σₛ',ᵣ p(s', r | s, a) [ r + γ v*(s') ]
q*(s, a) = Σₛ',ᵣ p(s', r | s, a) [ r + γ maxₐ' q*(s', a') ]
```

In the backup-diagram picture, the branch over actions at the root is no longer averaged;
it is maxed (drawn as an arc across the action branches).

> **🔑 KEY IDEA** · **Optimal values make optimal behaviour easy**
>
> If you know v*, act greedily with a one-step lookahead: π*(s) = argmaxₐ Σ p [r + γ v*(s')].
> If you know q*, you do not even need the model: π*(s) = argmaxₐ q*(s, a). That is why
> Q-learning learns q* rather than v*.

Greedy with respect to v* is optimal even though it only looks one step ahead, because v*
already accounts for every step beyond it. Long-term consequences have been folded into a
local quantity.

In [ ]:
#@title 🧠 **Knowledge check 4** · +25 XP — run, then pick an answer
rlcolab.quiz("03-mdps#3", {"q": "What makes the Bellman optimality equation harder to solve than the expectation equation?", "options": ["It has more unknowns", "The max makes it nonlinear, so one linear solve no longer works", "It requires γ < 1", "It is only defined for deterministic MDPs"], "answer": 1, "explain": "Same number of unknowns, but max is nonlinear. We need iterative schemes such as value iteration and policy iteration, which is the topic of the next lesson."})

### Finding π* the hard way

For a small MDP you can find π* without any clever algorithm: there are |A|^|S| deterministic
policies, and you can evaluate each one exactly. A 3 × 3 grid with 6 decision states has
4⁶ = 4096. The 4 × 3 grid above already has 4⁹ = 262 144, and a 10 × 10 grid has about
4¹⁰⁰ ≈ 10⁶⁰. Do it once anyway: seeing the winner satisfy the
optimality equation and dominate every other policy in **every** state makes the theory
concrete.

---
### 🧪 Exercise 3 · Optimal policies by brute force · ⭐ **120 XP**

A finite MDP has finitely many deterministic policies, so in principle you can just try them all. Do that (once!) to see the optimality theory with your own eyes.

1. `brute_force_optimal(env, gamma)` — loop over every assignment of an action to each state in `decision_states(env)` (`itertools.product`), turn it into a one-hot policy `np.eye(4)[actions]`, evaluate it exactly with the provided `solve_v`, and return `(actions, V)` of the policy with the **largest sum of state values**.
2. `is_optimal(env, V, gamma, tol)` — return True iff `V(s) = maxₐ q(s, a)` (within `tol`) in every state, using the provided `q_from_v`. That is the Bellman optimality equation.

The check confirms that the winner equals v*, satisfies Bellman optimality, and dominates every other policy **in every state**. **Run** shows the optimal policy and the distribution of v(start) over all 4096 policies.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 03-mdps/03-optimal-policy
import itertools
import numpy as np
import rlhud
from rlenvs import GridWorld


# ---- helpers from the previous two exercises (complete) ----------------------
def q_from_v(env, V, s, gamma):
    q = np.zeros(env.n_actions)
    for a in range(env.n_actions):
        for prob, s2, r, done in env.P[s][a]:
            q[a] += prob * (r + (0.0 if done else gamma * V[s2]))
    return q


def solve_v(env, policy, gamma):
    """Exact v_π for policy[s, a] = π(a|s) via (I − γ P_π) v = r_π."""
    n = env.n_states
    P_pi, r_pi = np.zeros((n, n)), np.zeros(n)
    for s in range(n):
        for a in range(env.n_actions):
            if policy[s, a] == 0:
                continue
            for prob, s2, r, done in env.P[s][a]:
                r_pi[s] += policy[s, a] * prob * r
                if not done:
                    P_pi[s, s2] += policy[s, a] * prob
    return np.linalg.solve(np.eye(n) - gamma * P_pi, r_pi)


def decision_states(env):
    """States where the action matters (not terminal, not a wall)."""
    return [s for s in range(env.n_states) if s not in env.terminal_states and s not in env.wall_states]


# ---- your code ------------------------------------------------------------------
def brute_force_optimal(env, gamma):
    """Evaluate EVERY deterministic policy exactly and keep the best one.

    A deterministic policy assigns one action to each decision state (states that are
    terminal or walls can use action 0). Compare policies by the SUM of their state
    values; the optimal policy is at least as good in every state, so it also wins the sum.

    Returns (actions, V): actions is an int array of length n_states, V its exact value.
    """
    states = decision_states(env)
    best_actions, best_v, best_score = None, None, -np.inf
    # TODO: loop over itertools.product(range(env.n_actions), repeat=len(states)),
    #       build the action array and its one-hot policy, evaluate it with solve_v,
    #       and keep the policy with the largest v.sum()
    raise NotImplementedError
    return best_actions, best_v


def is_optimal(env, V, gamma, tol=1e-8):
    """Bellman optimality test: V(s) == max_a q(s, a) in every state (within tol)."""
    # TODO: compare V[s] with the best one-step lookahead value in every state
    raise NotImplementedError


if __name__ == "__main__":
    env = GridWorld(["..G",
                     ".#X",
                     "S.."], slip=0.2, step_reward=-0.04)
    gamma = 0.9
    n_pol = env.n_actions ** len(decision_states(env))
    actions, v_star = brute_force_optimal(env, gamma)
    rlhud.grid(env.to_grid(v_star), title=f"Best of all {n_pol} deterministic policies",
               policy=env.policy_grid(actions), walls=env.walls_mask(), labels=env.labels())

    v_rand = solve_v(env, np.full((env.n_states, 4), 0.25), gamma)
    rlhud.table([["optimal (brute force)", round(float(v_star[env.start]), 4), is_optimal(env, v_star, gamma)],
                 ["uniform random", round(float(v_rand[env.start]), 4), is_optimal(env, v_rand, gamma)]],
                headers=["policy", "v(start)", "satisfies Bellman optimality?"], title="Optimality test")

    # Distribution of v(start) across every deterministic policy
    states = decision_states(env)
    starts = []
    for choice in itertools.product(range(4), repeat=len(states)):
        a = np.zeros(env.n_states, dtype=int); a[states] = choice
        starts.append(solve_v(env, np.eye(4)[a], gamma)[env.start])
    hist, edges = np.histogram(starts, bins=12)
    rlhud.bars(hist, labels=[f"{e:.2f}" for e in edges[:-1]], title="v(start) over all deterministic policies",
               highlight=len(hist) - 1)

In [ ]:
#@title ✅ **Check** · Optimal policies by brute force (+120 XP) — tests the code you last ran above
rlcolab.check("03-mdps/03-optimal-policy")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

`decision_states(env)` gives the list of states that need an action. `itertools.product(range(env.n_actions), repeat=len(states))` yields every tuple of actions for them.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

For each tuple: `actions = np.zeros(env.n_states, dtype=int); actions[states] = choice; v = solve_v(env, np.eye(env.n_actions)[actions], gamma)`. Keep it if `v.sum()` beats the best so far.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

`is_optimal`: `return all(abs(V[s] - q_from_v(env, V, s, gamma).max()) <= tol for s in range(env.n_states))`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("03-mdps/03-optimal-policy")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 03-mdps/03-optimal-policy
# Reference solution
import itertools
import numpy as np
import rlhud
from rlenvs import GridWorld


# ---- helpers from the previous two exercises (complete) ----------------------
def q_from_v(env, V, s, gamma):
    q = np.zeros(env.n_actions)
    for a in range(env.n_actions):
        for prob, s2, r, done in env.P[s][a]:
            q[a] += prob * (r + (0.0 if done else gamma * V[s2]))
    return q


def solve_v(env, policy, gamma):
    """Exact v_π for policy[s, a] = π(a|s) via (I − γ P_π) v = r_π."""
    n = env.n_states
    P_pi, r_pi = np.zeros((n, n)), np.zeros(n)
    for s in range(n):
        for a in range(env.n_actions):
            if policy[s, a] == 0:
                continue
            for prob, s2, r, done in env.P[s][a]:
                r_pi[s] += policy[s, a] * prob * r
                if not done:
                    P_pi[s, s2] += policy[s, a] * prob
    return np.linalg.solve(np.eye(n) - gamma * P_pi, r_pi)


def decision_states(env):
    """States where the action matters (not terminal, not a wall)."""
    return [s for s in range(env.n_states) if s not in env.terminal_states and s not in env.wall_states]


# ---- your code ------------------------------------------------------------------
def brute_force_optimal(env, gamma):
    """Evaluate EVERY deterministic policy exactly and keep the best one.

    A deterministic policy assigns one action to each decision state (states that are
    terminal or walls can use action 0). Compare policies by the SUM of their state
    values; the optimal policy is at least as good in every state, so it also wins the sum.

    Returns (actions, V): actions is an int array of length n_states, V its exact value.
    """
    states = decision_states(env)
    best_actions, best_v, best_score = None, None, -np.inf
    for choice in itertools.product(range(env.n_actions), repeat=len(states)):
        actions = np.zeros(env.n_states, dtype=int)
        actions[states] = choice
        v = solve_v(env, np.eye(env.n_actions)[actions], gamma)
        if v.sum() > best_score + 1e-12:
            best_actions, best_v, best_score = actions, v, v.sum()
    return best_actions, best_v


def is_optimal(env, V, gamma, tol=1e-8):
    """Bellman optimality test: V(s) == max_a q(s, a) in every state (within tol)."""
    return all(abs(V[s] - q_from_v(env, V, s, gamma).max()) <= tol for s in range(env.n_states))


if __name__ == "__main__":
    env = GridWorld(["..G",
                     ".#X",
                     "S.."], slip=0.2, step_reward=-0.04)
    gamma = 0.9
    n_pol = env.n_actions ** len(decision_states(env))
    actions, v_star = brute_force_optimal(env, gamma)
    rlhud.grid(env.to_grid(v_star), title=f"Best of all {n_pol} deterministic policies",
               policy=env.policy_grid(actions), walls=env.walls_mask(), labels=env.labels())

    v_rand = solve_v(env, np.full((env.n_states, 4), 0.25), gamma)
    rlhud.table([["optimal (brute force)", round(float(v_star[env.start]), 4), is_optimal(env, v_star, gamma)],
                 ["uniform random", round(float(v_rand[env.start]), 4), is_optimal(env, v_rand, gamma)]],
                headers=["policy", "v(start)", "satisfies Bellman optimality?"], title="Optimality test")

    # Distribution of v(start) across every deterministic policy
    states = decision_states(env)
    starts = []
    for choice in itertools.product(range(4), repeat=len(states)):
        a = np.zeros(env.n_states, dtype=int); a[states] = choice
        starts.append(solve_v(env, np.eye(4)[a], gamma)[env.start])
    hist, edges = np.histogram(starts, bins=12)
    rlhud.bars(hist, labels=[f"{e:.2f}" for e in edges[:-1]], title="v(start) over all deterministic policies",
               highlight=len(hist) - 1)

## 06 · Pitfalls

- **Bootstrapping through termination.** If a transition ends the episode, its target is just
  r, never r + γ·V(s'). Forgetting this is the most common bug in every algorithm from here to PPO.
- **Mixing up p(s'|s,a) and p(s',r|s,a).** In GridWorld one outcome tuple already pairs a next
  state with its reward; just weight each tuple by its probability.
- **Evaluating with the wrong policy.** v_π depends on π. The values of the random policy
  are not "the values of the grid".
- **γ = 1 with a policy that never terminates** makes I − γP_π singular: the return is
  infinite (or undefined). Use γ < 1 or guarantee termination.

## 07 · What carries forward

| Concept | Where it returns |
|---|---|
| The model p, stored in env.P | Dynamic programming (next lesson), Dyna-Q's learned model |
| One-step lookahead | Policy improvement, value iteration, every greedy policy |
| Bellman expectation equation | Policy evaluation, TD(0), SARSA, critics in actor-critic |
| Bellman optimality equation | Value iteration, Q-learning, DQN's target |
| Markov state design | Frame stacking, recurrent policies, feature engineering |

The linear solve worked because the policy was fixed. Next you will solve for the *best*
policy, iteratively, with **dynamic programming**.

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("03-mdps")